# Kraken no reconhecimento de manuscritos históricos em português

Este notebook apenas orquestra os módulos Python. Execute primeiro os controles, depois os tratamentos individuais e, por último, as combinações. A única métrica de qualidade é a taxa de erro por palavra (WER). As referências foram extraídas do DOCX fornecido e ainda precisam de revisão; portanto, os resultados são provisórios.

Execute a partir da raiz do projeto. No Colab, envie e extraia o ZIP do projeto e defina o diretório de trabalho como `/content/test_kraken` antes da primeira célula. A instalação das dependências e o download do modelo são automáticos. O notebook não contém imagens nem pesos incorporados. A implementação permanece nos arquivos `.py`.

Por padrão, são avaliadas as 12 imagens. As três etapas executam 17 variantes por imagem (204 inferências), sem repetir controles. Para conferir rapidamente o funcionamento, use `[images[6]]` na seleção de páginas. A avaliação da configuração exportada acrescenta uma variante por imagem.


In [ ]:
from kraken_pipeline.setup import configure_notebook, ensure_dependencies

configure_notebook()
ensure_dependencies()

## 1. Download, configuração e inicialização

O modelo padrão é o **PP-OCRv6 medium** (aproximadamente 64 MB), do [Zenodo](https://zenodo.org/records/21788410). A função baixa os pesos dentro do Colab, verifica o MD5 publicado e reutiliza o cache validado. É um modelo multilíngue com português no treinamento; não se afirma que seja específico de pt-BR.

Na comparação nas mesmas três páginas de ajuste e com as mesmas linhas, o medium teve WER de 65,45% na imagem original, o small 68,65% e o TraPrInq 80,82%. Os números não são uma avaliação geral dos modelos e o GT permanece provisório. O TraPrInq continua disponível em `download_portuguese_model()`.

O dispositivo é selecionado automaticamente: `cuda:0` com GPU disponível e `cpu` caso contrário. Se a instalação solicitar reinício, reinicie a sessão uma vez e execute novamente desde a primeira célula. Os arquivos e pacotes instalados são mantidos.


In [ ]:
from kraken_pipeline.config import PipelineConfig, PreprocessingConfig, ScoringConfig, RAW, NEURAL_BASELINE, INDIVIDUAL, COMBINED
from kraken_pipeline.engine import initialize_kraken
from kraken_pipeline.model_management import download_multilingual_model
from kraken_pipeline.experiments import dataset_images, run_experiments
from kraken_pipeline.reporting import review_reference, show_page, show_comparison, show_transcriptions, compare_runs, show_stages, export_report
from kraken_pipeline.setup import environment_info, default_device

recognition_model = download_multilingual_model(variant="medium")

config = PipelineConfig(
    recognition_model=recognition_model,
    device=default_device(),
    preprocessing=PreprocessingConfig(),
    scoring=ScoringConfig(lowercase=False, strip_punctuation=False),
)
environment_info()

In [ ]:
engine = initialize_kraken(config)
images = dataset_images(config)
selected_images = images  # Todas as 12 imagens; use [images[6]] para um teste rápido.

## 2. Revisão da transcrição de referência

Compare o texto fornecido com a imagem original. As possíveis discrepâncias estão em `ground_truth/comparison_review.md`. Corrija os arquivos `.txt` sem usar as previsões como referência e marque cada página integralmente conferida como `reviewed` em `review.json`. O DOCX original é preservado.


In [ ]:
review_reference(selected_images[0], config.reference_dir)

## 3. Controle original e baseline neural nativa

`raw`: imagem original → segmentação neural → reconhecimento.

`kraken_neural` (baseline): imagem original → binarização nativa nlbin (limiar 0,5) → segmentação neural → reconhecimento. O segmentador é o nativo do Kraken, selecionado na CLI por `segment -bl`. O reconhecedor e a política de WER são iguais nos dois controles; assim a comparação isola a binarização. O segmentador legado não é executado.

Inspecione os overlays. Zero linhas e texto vazio são registrados e contam como erro no WER. Cada tratamento individual começa novamente na original, prepara os pixels e depois executa segmentação e reconhecimento. Dewarp usa uma segmentação auxiliar para estimar curvatura antes da segmentação final.


In [ ]:
raw_run = run_experiments(config, engine, (RAW, NEURAL_BASELINE), selected_images, fail_fast=True)
raw_summary = show_comparison(raw_run)
show_transcriptions(raw_run, config.reference_dir)
for image in selected_images:
    show_stages(raw_run, image.stem)
export_report(raw_run, config.reference_dir)
show_page(raw_run, selected_images[0].stem)
show_page(raw_run, selected_images[0].stem, variant="kraken_neural")

## 4. Tratamentos individuais

Cada tratamento começa nos pixels originais e termina com nlbin nativo e segmentação neural, usando o mesmo reconhecedor da baseline do README. A correção de inclinação (`deskew`) e a correção de curvatura (`dewarp`) ocorrem antes da binarização. O dewarp usa uma segmentação neural auxiliar apenas para estimar a curvatura.

Otsu é um tratamento separado; o nlbin pode manter uma imagem que já seja bitonal. Não presuma que aumentar o contraste reduz o WER: cada tratamento precisa ser comparado individualmente.


In [ ]:
individual_run = run_experiments(config, engine, INDIVIDUAL, selected_images)
individual_summary = compare_runs((raw_run, individual_run))
show_transcriptions(individual_run, config.reference_dir)
for image in selected_images:
    show_stages(individual_run, image.stem)
export_report(individual_run, config.reference_dir)

## 5. Combinações ordenadas

Analise os resultados individuais antes de escolher as combinações em `config.py`. A tabela completa reúne as etapas anteriores e as novas combinações, sem refazer o OCR dos controles e tratamentos individuais. A comparação verifica hashes de páginas, modelos, referências e código, além da política de WER e da semente. As operações de cada combinação são aplicadas na ordem definida na tupla.

Quando as mesmas páginas orientam a escolha dos tratamentos, as pontuações são exploratórias e não estimam o desempenho em páginas independentes.


In [ ]:
combined_run = run_experiments(config, engine, COMBINED, selected_images)
combined_summary = compare_runs((raw_run, individual_run, combined_run))
show_transcriptions(combined_run, config.reference_dir)
for image in selected_images:
    show_stages(combined_run, image.stem)
export_report(combined_run, config.reference_dir)

## Interpretação dos resultados

WER = (substituições + deleções + inserções) / número de palavras da referência. Quanto menor, melhor; valores acima de 100% são possíveis. O WER do conjunto soma erros e palavras de referência, em vez de calcular a média dos percentuais das páginas.

Uma diferença negativa indica melhora em relação ao controle neural (`kraken_neural`). Variantes incompletas não recebem pontuação agregada. A tabela informa quantas páginas produziram texto. Zero linhas e reconhecimento vazio são registrados explicitamente; uma saída vazia não significa acerto de 100%. As pastas de saída contêm configuração, hashes, versões dos pacotes, situação das referências, imagens, sobreposições, transcrições e resultados.

Ambos os notebooks foram executados no Colab T4. A busca avaliada não demonstrou melhoria nas páginas reservadas; as referências continuam provisórias. Consulte `vault/10_validacao_remota_colab.md` para a metodologia e os limites da validação.


## 6. Reutilização dos resultados do ajuste

Execute `kraken_tuning.ipynb` separadamente, copie o novo `best.json` neural para este projeto e informe seu caminho abaixo. Perfis antigos com segmentação legada são incompatíveis e precisam de nova busca; suas métricas históricas não se aplicam à versão neural. A função de carregamento verifica os pesos do reconhecedor e a política de pontuação. Os parâmetros do nlbin da baseline permanecem nos valores padrão, mesmo quando o limiar selecionado é diferente.

A avaliação nas páginas usadas para ajuste é exploratória. Use a etapa de páginas reservadas no notebook de ajuste para uma verificação independente. Não é necessário executar os dois notebooks simultaneamente.


In [ ]:
from pathlib import Path
from kraken_pipeline.tuning import load_tuned_config

BEST_RESULTS = None  # Informe somente um best.json de uma nova busca neural.
if BEST_RESULTS is not None:
    tuned_config, tuned_variant = load_tuned_config(config, Path(BEST_RESULTS))
    tuned_run = run_experiments(tuned_config, engine, (tuned_variant,), selected_images)
    tuned_summary = compare_runs((raw_run, tuned_run))
    show_transcriptions(tuned_run, config.reference_dir)
    for image in selected_images:
        show_stages(tuned_run, image.stem)
    export_report(tuned_run, config.reference_dir)